# 01 — Accessing the API

Модулі курсу: **Accessing the API → Getting an API key → Making a Request**

Перший робочий контакт з Claude API: setup, перший запит, розбір відповіді.

## Setup

Залежності вже встановлені через `pip install -r requirements.txt` (з кореня проєкту).
Ключ — у `.env` (скопіюй `.env.example` → `.env`, встав свій `ANTHROPIC_API_KEY`).

In [ ]:
# Завантажуємо змінні середовища з файлу .env (там лежить твій API-ключ).
# load_dotenv() читає .env і робить його вміст доступним через os.getenv(...)
from dotenv import load_dotenv
load_dotenv()

# Перевіряємо, що ключ реально підвантажився. Якщо забув заповнити .env —
# краще одразу побачити зрозумілу помилку тут, ніж загадкову 401 Unauthorized пізніше
import os
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY порожній — заповни .env"

# Anthropic — клас клієнта з офіційного SDK.
# Він САМ бере ключ зі змінної середовища ANTHROPIC_API_KEY — передавати
# ключ вручну в код не треба (і не варто, щоб не залишити його в git-історії)
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"  # яку модель використовуємо в усіх запитах нижче

print("Клієнт готовий, модель:", model)

## Перший запит

`client.messages.create()` — обов'язкові аргументи: `model`, `max_tokens`, `messages`.

In [ ]:
# client.messages.create() — головна функція, яка й шле запит до Claude.
# 3 обов'язкові параметри:
#   model      — яку модель використати
#   max_tokens — ЛІМІТ довжини відповіді (safety cap, не "ціль для досягнення")
#   messages   — список повідомлень розмови (поки що одне, від юзера)
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[
        # role="user" означає, що це наш (людський) текст, а не відповідь моделі
        {"role": "user", "content": "What is quantum computing? Answer in one sentence"}
    ]
)

# Виводимо ВЕСЬ об'єкт відповіді — подивись, скільки там метаданих
# (id, role, model, content, stop_reason, usage...), не тільки сам текст
message

## Розбір відповіді

Повний об'єкт `message` містить багато метаданих — зазвичай потрібен тільки текст.

In [ ]:
# content — це СПИСОК блоків відповіді (може бути кілька блоків,
# наприклад текст + tool_use — про це буде окремий модуль курсу).
# [0].text дістає текст саме з першого (поки єдиного) блоку
print("Текст відповіді:")
print(message.content[0].text)

# usage показує, скільки токенів пішло на вхід (наш промпт) і на вихід
# (відповідь моделі) — знадобиться пізніше для підрахунку вартості запитів
print("\nUsage (input/output tokens):", message.usage)

# stop_reason пояснює, ЧОМУ модель зупинилась:
# "end_turn" — природно закінчила думку
# "max_tokens" — вперлась у ліміт max_tokens
# "stop_sequence" — зустріла задану стоп-фразу (буде окремий модуль)
print("Stop reason:", message.stop_reason)

## 🧪 Своя перевірка

Зміни `content` нижче на власне питання й запусти — переконайся, що розумієш кожен параметр.

In [ ]:
# Заміни текст "Твоє питання тут" на щось своє і запусти клітинку (Shift+Enter)
my_message = client.messages.create(
    model=model,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Твоє питання тут"}
    ]
)

print(my_message.content[0].text)